# Classification I: from labeled data to predictions
## Campus gym: who renews their membership?
**Salem Othman · Student notebook**

Last week you made data model-ready. Today you use it: you give a computer 150 members whose answer you already know, and it labels members it has never seen.

**The question:** will this member renew?

**Today's data:** `members`, 200 gym members. For each one you have four usage measurements and the answer, `renewed` (1 = renewed, 0 = did not).

By the end you will have a model you can defend, not just a model that runs.

### Today's six steps
| Part | Step | What you do | Exercise |
|---|---|---|---|
| 1 | Features and target | Split the table into X and y, and find the baseline | E1 |
| 2 | Train and test | Keep unseen rows for the final check | E2 |
| 3 | Neighbors | Fit KNN and compare training with test accuracy | E3 |
| 4 | Scaling | See why distance needs comparable units | E4 |
| 5 | Choosing k | Use cross-validation, not the test set | E5 |
| 6 | Audit | Fix AI-generated code that runs but is wrong | E6 |

**For every exercise:** predict → write the code → check → explain.

**How to work**
- Run cells from top to bottom with **Shift + Enter**.
- In an exercise cell, **write the code yourself** under each numbered step. Keep the variable names exactly as written; the Check cell looks for them.
- `PASS` means that step is done. `FIX` tells you what to look at.
- **Challenge tasks** are optional. Do them if you finish early or you have taken Machine Learning.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
RANDOM_STATE = 42

def check(label, passed):
    """Print PASS or FIX for one requirement."""
    print(("PASS  " if passed else "FIX   ") + label)

def need(value, step):
    """Stop with a clear message if an earlier step is not finished."""
    if value is None:
        raise RuntimeError(f"Finish {step} first, then run this cell again.")

print("Setup complete.")

Setup complete.


# Part 1 · Features and target
The gym has 200 members from last year. For each one you know how they used the gym and whether they renewed.

| Column | Meaning |
|---|---|
| `member_id` | Membership number. Useful for tracing a row, never a model input. |
| `visits_per_month` | Average visits per month |
| `avg_minutes` | Average minutes per visit |
| `months_member` | How long they had been a member |
| `distance_km` | Distance from home to the gym |
| `renewed` | **The answer:** 1 = renewed, 0 = did not renew |

**Words we will use all term**
- **Features (X):** what you measure. Here, the four usage columns.
- **Target (y):** what you want to predict. Here, `renewed`.
- **Labeled record:** one member whose answer you already know.
- **Binary classification:** the target has two classes. With three or more it is multiclass.

Classification predicts a **label**. Last week's rent question would be **regression**, because rent is a number. If nobody had told us who renewed, we could only **cluster** members into groups.

In [2]:
rng = np.random.default_rng(7)
n_members = 200
renewed = rng.binomial(1, 0.62, n_members)


def usage(mean_yes, sd_yes, mean_no, sd_no, low, high, decimals=0):
    """Make one usage column: renewing members differ from leaving members."""
    values = np.where(
        renewed == 1,
        rng.normal(mean_yes, sd_yes, n_members),
        rng.normal(mean_no, sd_no, n_members),
    )
    return np.round(np.clip(values, low, high), decimals)


members = pd.DataFrame(
    {
        "member_id": [f"M{i:03d}" for i in range(1, n_members + 1)],
        "visits_per_month": usage(12, 4, 5, 3, 0, 25).astype(int),
        "avg_minutes": usage(55, 15, 42, 15, 15, 120).astype(int),
        "months_member": usage(18, 10, 9, 7, 1, 48).astype(int),
        "distance_km": usage(3, 2, 6, 3, 0.2, 20, 1),
        "renewed": renewed,
    }
)

feature_cols = ["visits_per_month", "avg_minutes", "months_member", "distance_km"]
print("Rows, columns:", members.shape)
display(members.head())

Rows, columns: (200, 6)


,member_id,visits_per_month,avg_minutes,months_member,distance_km,renewed
0,M001,8,49,8,5.5,0
1,M002,0,32,8,5.1,0
2,M003,3,15,15,4.6,0
3,M004,7,30,22,0.6,1
4,M005,15,64,22,2.6,1


### E1 · Features, target, and the baseline
**Predict:** if you had to guess one answer for every member without looking at their usage, what would you guess, and how often would you be right?

**The baseline** is the accuracy of always predicting the most common class. A model that cannot beat it has taught you nothing.

I would guess that the model would be accurate in predicting the renewed feature ~75% of the time. 

In [3]:
X = None
y = None
class_counts = None
baseline_accuracy = None

# Write your code below. Your code must create the variables set to None above.

# 1. X: the four measurement columns of members (not member_id, not renewed).
#    y: the renewed column.
#    Hint: members[feature_cols] and members["renewed"]

# 2. class_counts: how many members renewed and how many did not.
#    Hint: .value_counts()

# 3. baseline_accuracy: the share of the most common class.
#    This is what you get by predicting that one class for everybody.
#    Hint: class_counts.max() / len(members)

# 4. Print class_counts and baseline_accuracy.

X = members[feature_cols]
y = members["renewed"]
class_counts = y.value_counts()
baseline_accuracy = class_counts.max() / len(members)

print(f"Class Counts: \n{class_counts.to_dict()}\n\n")

print(f"Baseline Accuracy: {baseline_accuracy}")

Class Counts: 
{1: 124, 0: 76}


Baseline Accuracy: 0.62


**Check E1**

In [4]:
need(X, "E1"); need(y, "E1"); need(class_counts, "E1"); need(baseline_accuracy, "E1")
check("X holds the four usage columns, and nothing else", list(X.columns) == feature_cols)
check("member_id and renewed are not features", not {"member_id", "renewed"} & set(X.columns))
check("y is the renewed column", y.equals(members["renewed"]))
check("class_counts covers every member", class_counts.sum() == len(members))
check("baseline_accuracy is the share of the most common class",
      np.isclose(baseline_accuracy, members["renewed"].value_counts().max() / len(members)))
check("the baseline is above half (the classes are not balanced)", baseline_accuracy > 0.5)

PASS  X holds the four usage columns, and nothing else
PASS  member_id and renewed are not features
PASS  y is the renewed column
PASS  class_counts covers every member
PASS  baseline_accuracy is the share of the most common class
PASS  the baseline is above half (the classes are not balanced)


**Write:** what would a model with 60% accuracy tell you about this data?

*Your answer:* We can predict if a user renews with an accuracy of 60%

# Part 2 · Train and test
A model that memorizes every member scores 100% on the members it memorized and can still be useless on new ones. The only fair check is data the model has never seen.

- **Training rows:** the model learns from these.
- **Test rows:** locked away until the end, used once.
- **Generalization:** doing well on rows you did not train on. That is the whole point.

`stratify=y` keeps the same renewed/not-renewed share in both parts, so the test baseline stays comparable.

### E2 · Split the data
**Predict:** we have 200 members and keep 25% for testing. How many rows go to each part?

*Your prediction:*
150 (75%) rows go into training set, 50 (25%) go into testing set.

In [5]:
X_train = X_test = y_train = y_test = None
test_baseline = None

need(X, "E1")

# Write your code below. Your code must create the variables set to None above.

# 1. Split X and y into X_train, X_test, y_train, y_test.
#    Use test_size=0.25, random_state=RANDOM_STATE, and stratify=y.
#    Hint: train_test_split(X, y, test_size=..., random_state=..., stratify=...)

# 2. test_baseline: the share of the most common class in y_test.
#    Hint: y_test.value_counts().max() / len(y_test)

# 3. Print how many rows are in the training set and the test set, and print test_baseline.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
test_baseline = y_test.value_counts().max() / len(y_test)

print(f"Test Baseline: {test_baseline}\n")

Test Baseline: 0.62



**Check E2**

In [6]:
need(X_train, "E2"); need(test_baseline, "E2")
check("every member is in exactly one part", len(X_train) + len(X_test) == len(X))
check("the test part is a quarter of the members", len(X_test) == round(0.25 * len(X)))
check("no member is in both parts", not set(X_train.index) & set(X_test.index))
check("the split is stratified: both parts have a similar renewal share",
      abs(y_train.mean() - y_test.mean()) < 0.05)
check("test_baseline is the share of the most common class in y_test",
      np.isclose(test_baseline, y_test.value_counts().max() / len(y_test)))

PASS  every member is in exactly one part
PASS  the test part is a quarter of the members
PASS  no member is in both parts
PASS  the split is stratified: both parts have a similar renewal share
PASS  test_baseline is the share of the most common class in y_test


**Write:** why can we not judge a model on the rows it trained on?

*Your answer:* We cannot judge amodel on the rows it is trained on because it already knows the answer to the row. 

# Part 3 · How KNN decides
K-Nearest Neighbors has no formula to fit. At training time it just **keeps the training rows**. To label a new member it:
1. measures the distance from the new member to every training member,
2. takes the `k` closest ones,
3. gives the new member whatever label most of them have.

Distance is ordinary straight-line (Euclidean) distance. For two features:
`distance = sqrt((visits1 - visits2)**2 + (minutes1 - minutes2)**2)`

**Predict:** a new member visits 12 times a month and stays 45 minutes. Which of these two members is the nearer neighbour?

In [7]:
new_member = {"visits_per_month": 12, "avg_minutes": 45}
neighbours = pd.DataFrame({
    "member": ["P", "Q"],
    "visits_per_month": [2, 11],
    "avg_minutes": [44, 70],
    "renewed": [0, 1],
})

raw = np.sqrt((neighbours["visits_per_month"] - new_member["visits_per_month"]) ** 2
              + (neighbours["avg_minutes"] - new_member["avg_minutes"]) ** 2)
neighbours["raw_distance"] = raw.round(1)
display(neighbours)
print("Nearest neighbour on raw numbers:", neighbours.loc[raw.idxmin(), "member"])

,member,visits_per_month,avg_minutes,renewed,raw_distance
0,P,2,44,0,10.0
1,Q,11,70,1,25.0


Nearest neighbour on raw numbers: P


Member P is two visits a month; member Q is eleven, like our new member. Yet P wins on raw numbers, because minutes are measured in a range of about 15 to 120 and visits in a range of 0 to 25. **The feature with the bigger numbers decides the distance.** Part 4 fixes this.

### E3 · Fit KNN and compare training with test accuracy
**Predict:** what training accuracy will k=1 get? Why?

k=1 will get 100% training accuracy. Because the training points are its own nearest neighbor. 

In [8]:
knn1 = None
knn5 = None
raw_scores = None

need(X_train, "E2")

# Write your code below. Your code must create the variables set to None above.

# 1. knn1: a KNeighborsClassifier with n_neighbors=1, fitted on X_train and y_train.
#    Hint: KNeighborsClassifier(n_neighbors=1).fit(...)

knn1 = KNeighborsClassifier(n_neighbors=1).fit(X_train, y_train)

# 2. knn5: the same thing with n_neighbors=5.

knn5 = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)


# 3. raw_scores: a DataFrame with one row per model and these columns:
#       k, train_accuracy, test_accuracy
#    Use model.score(X_train, y_train) and model.score(X_test, y_test).
#    Hint: build a list of dictionaries, then pd.DataFrame(that list)

raw_scores = pd.DataFrame(
    [
        {
            "k": 1,
            "train_accuracy": knn1.score(X_train, y_train),
            "test_accuracy": knn1.score(X_test, y_test),
        },
        {
            "k": 5,
            "train_accuracy": knn5.score(X_train, y_train),
            "test_accuracy": knn5.score(X_test, y_test),
        },
    ]
)

# 4. Display raw_scores. Compare each number with the test baseline from E2.

display(raw_scores)

,k,train_accuracy,test_accuracy
0,1,1.000000,0.8
1,5,0.893333,0.8


**Check E3**

In [9]:
need(knn1, "E3"); need(knn5, "E3"); need(raw_scores, "E3")
check("knn1 uses one neighbour and knn5 uses five",
      knn1.n_neighbors == 1 and knn5.n_neighbors == 5)
check("both models were fitted on the training rows only", len(knn1._fit_X) == len(X_train))
check("raw_scores has one row per model, with k and both accuracies",
      set(raw_scores.columns) >= {"k", "train_accuracy", "test_accuracy"} and len(raw_scores) == 2)
check("the k=1 training accuracy is a perfect score",
      np.isclose(raw_scores.loc[raw_scores["k"] == 1, "train_accuracy"].iloc[0], 1.0))
check("every test accuracy is the real score of that model",
      np.isclose(raw_scores.loc[raw_scores["k"] == 5, "test_accuracy"].iloc[0], knn5.score(X_test, y_test)))

PASS  knn1 uses one neighbour and knn5 uses five
PASS  both models were fitted on the training rows only
PASS  raw_scores has one row per model, with k and both accuracies
PASS  the k=1 training accuracy is a perfect score
PASS  every test accuracy is the real score of that model


**Write:** k=1 scores 100% on the training rows but less on the test rows. What does that gap tell you?

*Your answer:* The gap tells me that the model was overfitting, it memorized the training data instead of its patterns. 

# Part 4 · Scaling
KNN adds up differences across features. `months_member` runs from 1 to 48, `distance_km` from 0.2 to 20, so they shout, while `visits_per_month` whispers. Scaling puts every feature on the same footing: `StandardScaler` subtracts the mean and divides by the standard deviation, so each feature has mean 0 and spread 1.

**The scaler learns those means and spreads from the training rows only.** A pipeline does that for you: `make_pipeline(StandardScaler(), KNeighborsClassifier(...))` fits the scaler on the training rows, then feeds the scaled rows to KNN. When you call `.score(X_test, y_test)`, it scales the test rows with the training numbers.

In [10]:
need(X_train, "E2")
train_spread = X_train.std(ddof=0).round(1)
print("Training spread (standard deviation) of each feature:")
display(train_spread)

scaled_distance = np.sqrt(((neighbours["visits_per_month"] - new_member["visits_per_month"]) / train_spread["visits_per_month"]) ** 2
                          + ((neighbours["avg_minutes"] - new_member["avg_minutes"]) / train_spread["avg_minutes"]) ** 2)
neighbours["scaled_distance"] = scaled_distance.round(2)
display(neighbours)
print("Nearest on raw numbers:", neighbours.loc[neighbours["raw_distance"].idxmin(), "member"],
      "| Nearest after scaling:", neighbours.loc[neighbours["scaled_distance"].idxmin(), "member"])

Training spread (standard deviation) of each feature:


visits_per_month     5.1
avg_minutes         15.0
months_member        9.6
distance_km          2.6
dtype: float64

,member,visits_per_month,avg_minutes,renewed,raw_distance,scaled_distance
0,P,2,44,0,10.0,1.96
1,Q,11,70,1,25.0,1.68


Nearest on raw numbers: P | Nearest after scaling: Q


Scaling changes **who the neighbours are**, so it changes the answer.

### E4 · The same KNN, with scaling
**Predict:** will scaling change the training accuracy, the test accuracy, both, or neither?

scaling will change both test and training accuracy. 

In [11]:
knn5_scaled = None

need(X_train, "E2")

# Write your code below. Your code must create the variable set to None above.

# 1. knn5_scaled: a pipeline that first scales the features, then runs
#    KNeighborsClassifier(n_neighbors=5). Fit it on X_train and y_train.
#    Hint: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(...)
#    The pipeline fits the scaler on the training rows only, which is what we want.

knn5_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(
    X_train, y_train
)

# 2. Print its training accuracy and its test accuracy, and compare them with the
#    unscaled k=5 numbers from E3.

print(f"Scaled n=5 training accuracy: {knn5_scaled.score(X_train, y_train):.4f}")
print(f"Scaled n=5 test accuracy: {knn5_scaled.score(X_test, y_test):.4f}")
print(f"Unscaled n=5 test accuracy: {knn5.score(X_test, y_test):.4f}")


Scaled n=5 training accuracy: 0.8867
Scaled n=5 test accuracy: 0.9200
Unscaled n=5 test accuracy: 0.8000


**Check E4**

In [12]:
need(knn5_scaled, "E4")
check("knn5_scaled is a pipeline with a scaler and a KNN step", len(knn5_scaled.steps) == 2)
check("it still uses five neighbours", knn5_scaled.steps[-1][1].n_neighbors == 5)
check("the scaler learned from the training rows only",
      np.allclose(knn5_scaled.steps[0][1].mean_, X_train.mean().values))
check("scaling improved the test accuracy for k=5",
      knn5_scaled.score(X_test, y_test) > knn5.score(X_test, y_test))

PASS  knn5_scaled is a pipeline with a scaler and a KNN step
PASS  it still uses five neighbours
PASS  the scaler learned from the training rows only
PASS  scaling improved the test accuracy for k=5


**Write:** which feature was drowning out the others before scaling, and how can you tell?

*Your answer:* The feature `avg_minutes` was drowning out the other features because it has a large range of nunbers in comparison to other features

# Part 5 · Choosing k
`k` controls how complex the model is.
- **k=1:** every training row wins its own patch of space. A complex, jagged boundary that follows noise. This is **overfitting**.
- **large k:** the vote is averaged over many members, so the boundary is smooth and may miss real structure. This is **underfitting**.

How do you choose? **Not** by trying every k on the test rows: do that and you have quietly fitted the test set, and your final number is no longer honest.

Use **cross-validation on the training rows**: split the training rows into 5 folds, train on 4 and score on 1, five times, and average. The test rows stay untouched until the end.

### E5 · A k table with cross-validation

In [13]:
k_table = None
best_k = None

need(X_train, "E2")
k_values = [1, 3, 5, 7, 9, 11, 15, 21]

# Write your code below. Your code must create the variable set to None above.

# 1. For every k in k_values, build the same scaled pipeline and collect:
#       k, train accuracy, cv_accuracy, test accuracy
#    cv_accuracy is the average of cross_val_score(pipeline, X_train, y_train, cv=5).
#    Cross-validation uses the TRAINING rows only.
#    Hint: loop over k_values, build the pipeline inside the loop, append a dictionary,
#    then k_table = pd.DataFrame(rows)

rows = []

for k in k_values:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    pipe.fit(X_train, y_train)
    cv_acc = round(cross_val_score(pipe, X_train, y_train, cv=5).mean(), 3)
    rows.append(
        {
            "k": k,
            "train_accuracy": pipe.score(X_train, y_train),
            "cv_accuracy": cv_acc,
            "test_accuracy": pipe.score(X_test, y_test),
        }
    )

k_table = pd.DataFrame(rows)
best_k = int(k_table.loc[k_table["cv_accuracy"].idxmax(), "k"])

# 2. Display k_table.

display(k_table)

# 3. best_k: the k with the highest cv_accuracy (not the highest test accuracy).
#    Hint: k_table.loc[k_table["cv_accuracy"].idxmax(), "k"

print(f"Best k: {best_k}")

,k,train_accuracy,cv_accuracy,test_accuracy
0,1,1.000000,0.800,0.84
1,3,0.913333,0.867,0.90
2,5,0.886667,0.860,0.92
3,7,0.906667,0.873,0.88
4,9,0.900000,0.880,0.90
5,11,0.893333,0.873,0.90
6,15,0.900000,0.867,0.90
7,21,0.880000,0.893,0.90


Best k: 21


**Check E5**

In [14]:
need(k_table, "E5"); need(best_k, "E5")
check("k_table has one row per k value", len(k_table) == len(k_values))
check("k_table reports train, cross-validation, and test accuracy",
      set(k_table.columns) >= {"k", "train_accuracy", "cv_accuracy", "test_accuracy"})
check("k=1 has the highest training accuracy of all the k values",
      k_table.loc[k_table["k"] == 1, "train_accuracy"].iloc[0] == k_table["train_accuracy"].max())
check("best_k was chosen by cross-validation, not by the test column",
      best_k == int(k_table.loc[k_table["cv_accuracy"].idxmax(), "k"]))
check("cross-validation used the training rows only",
      np.isclose(k_table.loc[k_table["k"] == best_k, "cv_accuracy"].iloc[0],
                 round(cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)),
                                       X_train, y_train, cv=5).mean(), 3)))

PASS  k_table has one row per k value
PASS  k_table reports train, cross-validation, and test accuracy
PASS  k=1 has the highest training accuracy of all the k values
PASS  best_k was chosen by cross-validation, not by the test column
PASS  cross-validation used the training rows only


**Write:** which k would you hand to the gym manager, and what evidence supports it? Name one thing that still worries you.

*Your answer:* I would give the gym manager k=21. It has the highest accuracy in testing and cv. The thing that worries me about it is the fact the training accuracy is lower than other options

### Challenge tasks (optional)
Do these only if you have finished E5 and want more. No solution is required for the exit ticket.
1. Add `weights="distance"` to the KNN step and rerun the k table. Does it change the best k?
2. Fit the best pipeline and print the members it gets wrong. Is there anything they have in common?
3. Repeat E2 and E5 with `random_state=1`. How much does the best k move? What does that tell you about reporting a single number?

# Part 6 · Audit the AI code
A student asks an AI tool for "the best model to predict renewals" and pastes back the code below. It runs, and it prints an accuracy of **1.00**: every test member labelled correctly, far above anything you produced today.

```python
# AI-generated code
data = members_extra
X_all = data.drop(columns=["member_id", "renewed"])

scaler = StandardScaler().fit(X_all)
X_scaled = scaler.transform(X_all)

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_scaled, data["renewed"], random_state=42)
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train2, y_train2)
print("Accuracy:", model.score(X_test2, y_test2))
```

`members_extra` is the same table plus one column pulled from the cancellation system.

**The question to answer before you touch the code: it runs, and a perfect 1.00 beats your model. Do you trust it?**

In [15]:
members_extra = members.copy()
cancel_rng = np.random.default_rng(11)
# The cancellation form only exists for members who already decided to leave.
members_extra["cancel_form_submitted"] = np.where(
    members_extra["renewed"] == 0, cancel_rng.binomial(1, 0.95, len(members_extra)), 0
)

display(
    members_extra.groupby("renewed")["cancel_form_submitted"]
    .mean()
    .rename("share who submitted the form")
)
display(members_extra.head())

renewed
0    0.947368
1    0.000000
Name: share who submitted the form, dtype: float64

,member_id,visits_per_month,avg_minutes,months_member,distance_km,renewed,cancel_form_submitted
0,M001,8,49,8,5.5,0,1
1,M002,0,32,8,5.1,0,1
2,M003,3,15,15,4.6,0,1
3,M004,7,30,22,0.6,1,0
4,M005,15,64,22,2.6,1,0


*Your answer: which problems can you find? (There are at least two.)*
1. The `cancel_form_submitted` feature is unnecissary due to it being recorded after the user does not renew.
2. The model was fit on X_all instead of X_train. This causes the testing set to be included in the model

### E6 · Fix it

In [16]:
fixed_features = None
Xf_train = Xf_test = yf_train = yf_test = None
fixed_model = None
fixed_accuracy = None

need(best_k, "E5")


# Write your code below. Your code must create the variables set to None above.

# 1. fixed_features: the feature columns the AI code should have used.
#    Drop the column that is only known after the member has already decided.
#    Hint: start from feature_cols
fixed_features = feature_cols.copy()

# 2. Split members_extra[fixed_features] and members_extra["renewed"] the same way as in E2
#    (test_size=0.25, random_state=RANDOM_STATE, stratify=y).
#    Name them Xf_train, Xf_test, yf_train, yf_test.
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    members_extra[fixed_features],
    members_extra["renewed"],
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=members_extra["renewed"],
)
# 3. fixed_model: the scaled pipeline with n_neighbors=best_k, fitted on the training rows.
fixed_model = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)
).fit(Xf_train, yf_train)
# 4. fixed_accuracy: its accuracy on the test rows. Print it next to the AI number (1.00).
fixed_accuracy = fixed_model.score(Xf_test, yf_test)

print("AI model accuracy (with leaked feature): 1.0000")
print(f"Fixed model accuracy (honest test):       {fixed_accuracy:.4f}")

AI model accuracy (with leaked feature): 1.0000
Fixed model accuracy (honest test):       0.9000


**Check E6**

In [17]:
need(fixed_features, "E6"); need(fixed_model, "E6"); need(fixed_accuracy, "E6")
check("the column that is only known after the decision was dropped",
      "cancel_form_submitted" not in fixed_features)
check("member_id and renewed are not features", not {"member_id", "renewed"} & set(fixed_features))
check("the split happens before anything is fitted (a pipeline does the scaling)",
      hasattr(fixed_model, "steps") and len(fixed_model.steps) == 2)
check("the model was fitted on the training rows only", len(Xf_train) == round(0.75 * len(members_extra)))
check("fixed_accuracy is the score on the untouched test rows",
      np.isclose(fixed_accuracy, fixed_model.score(Xf_test, yf_test)))
check("the honest accuracy is lower than the leaked perfect score", fixed_accuracy < 1.0)
check("it still beats the baseline", fixed_accuracy > test_baseline)

PASS  the column that is only known after the decision was dropped
PASS  member_id and renewed are not features
PASS  the split happens before anything is fitted (a pipeline does the scaling)
PASS  the model was fitted on the training rows only
PASS  fixed_accuracy is the score on the untouched test rows
PASS  the honest accuracy is lower than the leaked perfect score
PASS  it still beats the baseline


**Write:** in one or two sentences, how would you explain to the student who pasted that code why the perfect score was not real?

*Your answer:* A perfect score would not be possible because there is a column for cancel_form_submitted. This feature is only changed after the user has cancelled. 

# Exit ticket
Answer without running more code.
1. Our test baseline is 0.62 and a model scores 0.64. What would you say about that model?
2. Scaling moved the nearest neighbour from member P to member Q. In one sentence, why?
3. The AI code scored a perfect 1.00. Name the check you would run first, and what you expect to find.

*Your answers:*
1. The model did not learn much from the features. 0.64 is barely an improvement on 0.62
2. By scaling, we rescale all features to mean 0 with a standard deviation of 1, this prevents large distance calculations from dominating the result. 
3. I would check for leakage, some sort of error where the testing is included in the training. 

# Submit
- Restart the kernel and run all cells. Every check should print `PASS`.
- Every *Your answer* and *Your prediction* cell is filled in.
- Submit this notebook with its outputs.